In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os
from math import ceil
from scipy.stats import gaussian_kde
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import random


try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

In [2]:
# soma analytical data
somaAnaly = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/SomalogicAnalyteInfoV1_anonymized.csv')
somaAnaly.head()

,table_name,column_name,apt_name,seq_id,seq_id_version,soma_id,target_full_name,target,uni_prot,entrez_gene_id,entrez_gene_symbol,organism,units,type
0,Somalogic01V1_4ta,seq_10003_15,seq.10003.15,10003-15,3,SL019245,Zinc finger protein 41,ZNF41,P51814,7592,ZNF41,Human,RFU,Protein
1,Somalogic01V1_4ta,seq_10006_25,seq.10006.25,10006-25,3,SL019228,ETS domain-containing protein Elk-1,ELK1,P19419,2002,ELK1,Human,RFU,Protein
2,Somalogic01V1_4ta,seq_10008_43,seq.10008.43,10008-43,3,SL019234,Guanylyl cyclase-activating protein 1,GUC1A,P43080,2978,GUCA1A,Human,RFU,Protein
3,Somalogic01V1_4ta,seq_10010_10,seq.10010.10,10010-10,3,SL014943,Beclin-1,BECN1,Q14457,8678,BECN1,Human,RFU,Protein
4,Somalogic01V1_4ta,seq_10011_65,seq.10011.65,10011-65,3,SL019246,Inositol polyphosphate 5-phosphatase OCRL-1,OCRL,Q01968,4952,OCRL,Human,RFU,Protein


In [3]:
# sample source data
somaMeta = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/SomalogicMetaV1_anonymized.csv')
somaMeta.head()

/tmp/ipykernel_64939/3545570142.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  somaMeta = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/SomalogicMetaV1_anonymized.csv')


,sample_id,contributor_code,visit,plate_id,units,anml_fraction_used_0_005,anml_fraction_used_0_5,anml_fraction_used_20,anml_fraction_used_20_s1,anml_fraction_used_20_s2,...,hyb_control_norm_scale,norm_scale_0_005,norm_scale_0_5,norm_scale_20,norm_scale_20_s1,norm_scale_20_s2,norm_scale_20_s3,row_check,sample_matrix,sample_type
0,8a0e5419-20f3-44d6-bb46-bab165bf8d05,R,1,147,-1,0.866,0.836,0.847,-1.0,-1.0,...,0.934112,1.435652,0.824065,0.722277,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
1,1f2178f2-83a6-46df-aec1-3dd741621068,R,2,147,-1,0.936,0.947,0.954,-1.0,-1.0,...,0.884058,0.894184,0.804804,0.841661,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
2,40c3925c-a770-4adc-9a87-5d9e2a23b855,R,1,147,-1,0.973,0.939,0.913,-1.0,-1.0,...,0.906697,1.100648,1.043171,0.901319,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
3,9de3c941-b32b-4ef8-8d59-6b57b905c657,R,2,147,-1,0.888,0.843,0.859,-1.0,-1.0,...,0.994137,1.165921,0.942466,0.825704,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
4,03830809-3976-4f58-8279-ca2ffd0398c4,R,2,147,-1,0.850,0.828,0.804,-1.0,-1.0,...,0.925489,1.061411,0.659705,0.608147,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample


In [4]:
# sample data
somaData = {}
indices = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
for i in indices:
    file = f"../data/GNPC_Harmonized_Dataset_V1/Somalogic{i}V1_anonymized.csv"
    try:
        df = pd.read_csv(file, encoding="utf-8", engine="python")
    except:
        df = pd.read_csv(file, encoding="latin1", engine="python")

    somaData[i] = df

In [5]:
# clinical subject data
clinicalData = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/ClinicalV1_anonymized.csv')
clinicalData.head()

,contributor_code,sample_id,sequential_visit_number,age_at_visit,computed_age_range,sex,race,years_of_education,computed_years_education_range,computed_height_range,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,A,2d5cc071-c66c-4a10-b55b-00d6eb03b6f3,1,69,0,1,5,14,0,0,...,0,-1.0,8/25/2021,30,1,MMSE,-1,0,0,0
1,A,26ac68a0-91e9-4d7e-8b58-00e737c3bcd7,1,76,0,1,5,4,2,0,...,0,-1.0,9/23/2022,23,1,MMSE,1,1,0,0
2,A,28f4a373-ba8e-4832-8485-0158dfd8c62b,1,65,0,2,2,14,0,0,...,1,-1.0,1/13/2022,20,1,MMSE,1,1,0,0
3,A,64b27523-4353-420e-b01a-024c470ccf90,1,65,0,2,2,14,0,0,...,0,-1.0,12/10/2021,26,1,MMSE,2,1,0,0
4,A,b68baa17-b013-412d-bf71-0254b6945e8d,1,73,0,2,2,18,0,0,...,0,-1.0,9/28/2021,27,1,MMSE,-1,0,0,0


In [6]:
# clinical subject to sample mapping data
mappingData = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/PersonMappingV1_anonymized.csv')
mappingData.head()

,person_id,sample_id,is_somalogic,is_mass_spec
0,8aee12c9-962e-48f4-9d15-0008e4443c9d,8aee12c9-962e-48f4-9d15-0008e4443c9d,1,0
1,cd284063-9250-4149-9e8a-0009a615b59d,cd284063-9250-4149-9e8a-0009a615b59d,1,0
2,20fb319a-13a8-4284-83f6-000d1942beb2,20fb319a-13a8-4284-83f6-000d1942beb2,1,0
3,c0493c35-5a4f-4b34-8bdd-000e0c492ae1,c0493c35-5a4f-4b34-8bdd-000e0c492ae1,1,0
4,c0493c35-5a4f-4b34-8bdd-000e0c492ae1,7cdd93db-f393-4a53-a9cb-7cd67ef4b60b,1,0


In [7]:
def analy_filter(df_analytes):
    """
    Filter analytes for Human, Protein, and RFU units.
    Args:
        df_analytes (pd.DataFrame): DataFrame containing analyte information.
    Returns:
        pd.DataFrame: Filtered DataFrame with selected analytes.
    """
    # total number of rows
    total_rows = len(df_analytes)

    # apply selection criteria
    selected_df = df_analytes[
        (df_analytes["organism"] == "Human") &
        (df_analytes["type"] == "Protein") &
        (df_analytes["units"] == "RFU")
    ]

    # number of selected rows
    selected_rows = len(selected_df)

    total_rows, selected_rows

    print(f"Total rows: {total_rows}")
    print(f"Selected rows (Human, Protein, RFU): {selected_rows}")
    return selected_df

In [8]:
def soma_filter_by_analytes(somaData, analy_df):
    """
    Filter soma data by given analyte df.
    """
    filtered_somaData = {}
    for key, df in somaData.items():
        aptamer_cols = [c for c in df.columns if c.startswith("seq_")]
        table = f"Somalogic{key}V1_4ta"
        analy_ref = analy_df[analy_df["table_name"] == table]
        valid_cols = set(analy_ref["column_name"])
        filtered_aptamer_cols = [c for c in aptamer_cols if c in valid_cols]
        filtered_df = df[[c for c in df.columns if not c.startswith("seq_")] + filtered_aptamer_cols]
        print(f"For somaData {key}, Original columns: {len(aptamer_cols)}, Filtered columns: {len(filtered_aptamer_cols)}")
        filtered_somaData[key] = filtered_df
    return filtered_somaData

In [9]:
human_protein_analytes = analy_filter(somaAnaly)
human_protein_analytes.head()

Total rows: 7644
Selected rows (Human, Protein, RFU): 7289


,table_name,column_name,apt_name,seq_id,seq_id_version,soma_id,target_full_name,target,uni_prot,entrez_gene_id,entrez_gene_symbol,organism,units,type
0,Somalogic01V1_4ta,seq_10003_15,seq.10003.15,10003-15,3,SL019245,Zinc finger protein 41,ZNF41,P51814,7592,ZNF41,Human,RFU,Protein
1,Somalogic01V1_4ta,seq_10006_25,seq.10006.25,10006-25,3,SL019228,ETS domain-containing protein Elk-1,ELK1,P19419,2002,ELK1,Human,RFU,Protein
2,Somalogic01V1_4ta,seq_10008_43,seq.10008.43,10008-43,3,SL019234,Guanylyl cyclase-activating protein 1,GUC1A,P43080,2978,GUCA1A,Human,RFU,Protein
3,Somalogic01V1_4ta,seq_10010_10,seq.10010.10,10010-10,3,SL014943,Beclin-1,BECN1,Q14457,8678,BECN1,Human,RFU,Protein
4,Somalogic01V1_4ta,seq_10011_65,seq.10011.65,10011-65,3,SL019246,Inositol polyphosphate 5-phosphatase OCRL-1,OCRL,Q01968,4952,OCRL,Human,RFU,Protein


In [10]:
humanData = soma_filter_by_analytes(somaData, human_protein_analytes)

For somaData 01, Original columns: 624, Filtered columns: 540
For somaData 02, Original columns: 650, Filtered columns: 616
For somaData 03, Original columns: 650, Filtered columns: 629
For somaData 04, Original columns: 650, Filtered columns: 649
For somaData 05, Original columns: 650, Filtered columns: 641
For somaData 06, Original columns: 650, Filtered columns: 640
For somaData 07, Original columns: 650, Filtered columns: 647
For somaData 08, Original columns: 650, Filtered columns: 629
For somaData 09, Original columns: 650, Filtered columns: 573
For somaData 10, Original columns: 650, Filtered columns: 603
For somaData 11, Original columns: 599, Filtered columns: 527
For somaData 12, Original columns: 672, Filtered columns: 595


In [11]:
# Separate different sample matrices into different dataframes
somaGroups = {key: subdf for key, subdf in somaMeta.groupby("sample_matrix")}

In [12]:
# Extract sample data of a specific matrix type
def get_data_by_matrix_type(matrix_type, groups, somaData):
    if matrix_type not in groups:
        raise ValueError(f"Matrix type {matrix_type} not found in metadata.")
    
    sample_ids = groups[matrix_type]['sample_id'].tolist()
    data_frames = {}
    
    for idx, df in somaData.items():
        filtered_df = df[df['sample_id'].isin(sample_ids)]
        data_frames[idx] = filtered_df
    
    return data_frames

In [13]:
plasma = get_data_by_matrix_type("EDTA Plasma", somaGroups, humanData)
citrate = get_data_by_matrix_type("Citrate Plasma", somaGroups, humanData)
serum = get_data_by_matrix_type("Serum", somaGroups, humanData)
csf = get_data_by_matrix_type("CSF", somaGroups, humanData)

In [14]:
def merge_by_sample_id(dfs, id_col="sample_id"):
    # Start with the first df
    merged = dfs[0]

    for df in dfs[1:]:
        merged = merged.merge(df, on=id_col, how="outer", suffixes=("", "_dup"))

        # Remove columns ending in "_dup"
        dup_cols = [c for c in merged.columns if c.endswith("_dup")]
        merged = merged.drop(columns=dup_cols)

    return merged

def merge_left(df1, df2, id_col="sample_id"):
    merged = df1.merge(df2, on=id_col, how="left", suffixes=("", "_dup"))
    dup_cols = [c for c in merged.columns if c.endswith("_dup")]
    return merged.drop(columns=dup_cols)

def merge_spec(specs, indices, clinicalData):
    merged_spec = merge_by_sample_id([specs[idx] for idx in indices])
    merged_with_clinical = merge_left(merged_spec, clinicalData, id_col="sample_id")
    # Check if the two columns exist
    if "visit" not in merged_with_clinical.columns or "sequential_visit_number" not in merged_with_clinical.columns:
        pass

    # Check if values are identical
    elif (merged_with_clinical["visit"] == merged_with_clinical["sequential_visit_number"]).all():
        # Drop sequential_visit_number if identical
        merged_with_clinical = merged_with_clinical.drop(columns=["sequential_visit_number"])
        print("Dropped 'sequential_visit_number' because it is identical to 'visit'.")
    else:
        # Raise error if they differ
        mismatched_rows = merged_with_clinical[merged_with_clinical["visit"] != merged_with_clinical["sequential_visit_number"]]
        raise ValueError(
            f"'visit' and 'sequential_visit_number' differ in {len(mismatched_rows)} rows."
        )
    return merged_with_clinical

In [15]:
plasma_with_clinical = merge_spec(plasma, indices, clinicalData)
print(plasma_with_clinical.shape)
plasma_with_clinical.head()

Dropped 'sequential_visit_number' because it is identical to 'visit'.
(23000, 7344)


,sample_id,contributor_code,visit,sample_type,seq_10000_28,seq_10001_7,seq_10003_15,seq_10006_25,seq_10008_43,seq_10010_10,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,0003610c-e435-44f3-83b4-d40dd750a13f,C,1,Sample,457.2,385.2,149.8,719.2,403.7,307.5,...,-1,-1.0,11/18/2019,28,1,MMSE,-1,0,0,1
1,00090ab1-6e72-4b1f-8a38-c99dda26d7dd,G,1,Sample,500.7,292.5,205.2,959.3,547.0,317.5,...,-1,-1.0,1/1/1900,29,1,MMSE,-1,0,-1,-1
2,000a58d9-feea-4ff8-95d7-fca3fa91e174,B,1,Sample,487.0,313.6,158.4,553.7,453.7,299.2,...,0,0.0,11/4/2013,30,1,MMSE,0,0,1,0
3,000b14dc-ba73-46ee-a69e-e681ca3ad857,F,1,Sample,611.6,250.4,170.3,653.3,479.1,353.7,...,0,1.0,1/1/1900,-1,-1,MMSE,1,1,0,-1
4,001016d4-a28d-4a66-9c88-92b8a7a04d2d,F,1,Sample,602.3,254.2,189.0,638.8,452.4,463.5,...,0,1.0,1/1/1900,16,1,MMSE,1,1,0,-1


In [16]:
csf_with_clinical = merge_spec(csf, indices, clinicalData)
print(csf_with_clinical.shape)
csf_with_clinical.head()

Dropped 'sequential_visit_number' because it is identical to 'visit'.
(3233, 7344)


,sample_id,contributor_code,visit,sample_type,seq_10000_28,seq_10001_7,seq_10003_15,seq_10006_25,seq_10008_43,seq_10010_10,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,001facb2-bdfa-41d7-9feb-537a26784d4f,O,1,Sample,7.265798,7.120261,7.225679,7.853734,6.980599,6.293558,...,0,-1.0,1/1/1900,-1,-1,MOCA,-1,-1,-1,-1
1,0026f18a-c3f6-4255-a173-14ab2981006e,J,1,Sample,110.300000,164.000000,139.500000,116.200000,103.600000,-1.000000,...,0,0.0,1/1/1900,29,1,MOCA,0,0,-1,-1
2,002dcec8-19f1-483a-8389-76b9d847e5db,T,1,Sample,144.500000,170.100000,153.800000,105.000000,126.600000,144.500000,...,-1,-1.0,1/1/1900,-1,-1,MMSE,3,-1,-1,-1
3,002e0ec7-25f0-4ae8-84f3-3cc3579f883d,O,1,Sample,6.793472,7.744079,6.168176,6.732345,5.909490,6.047303,...,0,-1.0,1/1/1900,-1,-1,MOCA,-1,-1,-1,-1
4,003247db-359d-43b8-9398-e73337a8ef88,Q,1,Sample,152.400000,210.000000,160.800000,96.800000,135.000000,150.700000,...,-1,0.5,1/1/1900,-1,-1,MMSE,2,1,0,0


In [17]:
citrate_with_clinical = merge_spec(citrate, indices, clinicalData)
print(citrate_with_clinical.shape)
citrate_with_clinical.head()

Dropped 'sequential_visit_number' because it is identical to 'visit'.
(638, 7344)


,sample_id,contributor_code,visit,sample_type,seq_10000_28,seq_10001_7,seq_10003_15,seq_10006_25,seq_10008_43,seq_10010_10,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,000be381-ebad-42c6-bf86-28413e9226f8,E,1,Sample,1481.2,1324.6,253.9,820.5,381.8,255.5,...,-1,0.0,1/1/1900,-1,-1,MMSE,0,0,0,0
1,00448953-75bf-45b9-a2f3-59fa94f20246,E,2,Sample,497.4,337.4,183.1,675.5,413.7,353.8,...,-1,0.0,1/1/1900,-1,-1,MMSE,0,0,0,0
2,00711b3a-4bfc-499a-9cf0-5c607944d9d1,E,2,Sample,465.8,952.5,174.3,663.3,399.1,280.3,...,-1,0.0,1/1/1900,-1,-1,MMSE,0,0,0,0
3,0124fb54-dd31-4e8c-ab8d-79caae176db8,E,1,Sample,474.4,959.1,181.6,599.8,367.4,278.6,...,0,0.0,1/1/1900,-1,-1,MMSE,0,0,0,0
4,012aeb33-1227-4de8-b270-6d92e42c2922,E,1,Sample,530.0,526.3,198.6,621.3,355.3,285.5,...,0,0.0,1/1/1900,-1,-1,MMSE,0,0,0,0


In [18]:
serum_with_clinical = merge_spec(serum, indices, clinicalData)
print(serum_with_clinical.shape)
serum_with_clinical.head()

Dropped 'sequential_visit_number' because it is identical to 'visit'.
(4212, 7344)


,sample_id,contributor_code,visit,sample_type,seq_10000_28,seq_10001_7,seq_10003_15,seq_10006_25,seq_10008_43,seq_10010_10,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,00015dde-0136-4071-b39a-cdec57ac2bd7,U,1,Sample,1415.3,426.6,281.0,912.8,1017.4,653.9,...,-1,-1.0,1/1/1900,25,1,MOCA,3,0,-1,-1
1,0005609d-33b6-4ee2-9949-31456e9b9968,U,3,Sample,1307.5,1323.9,793.9,898.0,773.3,737.2,...,-1,-1.0,1/1/1900,23,1,MOCA,3,0,-1,-1
2,000b1705-9d8d-4bc3-9778-5040299f262d,U,4,Sample,1371.3,538.9,274.3,1018.1,1088.6,637.2,...,-1,-1.0,1/1/1900,29,1,MOCA,3,0,-1,-1
3,0011ef1d-08ef-4e0f-bdff-df0ac2407a86,U,3,Sample,1331.8,643.0,470.7,1003.8,1133.7,658.9,...,-1,-1.0,1/1/1900,27,1,MOCA,3,0,-1,-1
4,001a1a09-5289-45da-83e6-21334d20467f,U,3,Sample,1402.3,563.2,292.7,1071.0,1104.1,609.2,...,-1,-1.0,1/1/1900,26,1,MOCA,3,0,-1,-1


In [19]:
def longitudinal_stats(df, mappingData, converter=False, detail=False):
    df = df[df["age_at_visit"] != -1]

    # Merge dfB with dfA to bring person_id into dfB
    df_merged = df.merge(mappingData, on="sample_id", how="inner")

    # Group by person_id and generate list of sub-dataframes
    subdfs = [subdf for _, subdf in df_merged.groupby("person_id")]
    subdfs_multi_visit = [subdf for subdf in subdfs if subdf["visit"].max() > 1 and len(subdf) > 1]
    cohorts = []
    for df in subdfs_multi_visit:
        cohorts.append(df["contributor_code"].iloc[0])
        if df["contributor_code"].nunique() > 1:
            raise ValueError("Cohort inconsistency within the same person_id.")

    if converter:
        converted_subdfs_multi_visit = [df for df in subdfs_multi_visit if df["ad"].nunique() > 1 or df["ftd"].nunique() > 1 or df["pd"].nunique() > 1 or df["als"].nunique() > 1 or df["mci_sci"].nunique() > 1 or df["recruited_control"].nunique() > 1]

    print(f"\nTotal unique persons: {len(subdfs)}")
    print(f"Total persons with multiple visits: {len(subdfs_multi_visit)}")
    print(f"Total number of multiple visits: {sum(len(subdf) for subdf in subdfs_multi_visit)}")
    print("Different cohorts in multi-visit persons:", set(cohorts))
    if converter:
        print(f"Total persons with diagnosis changes over visits: {len(converted_subdfs_multi_visit)}")
    if detail:
        for i, subdf in enumerate(subdfs_multi_visit):
            print(f"\nPerson ID: {subdf['person_id'].iloc[0]} - Number of Visits: {len(subdf)}")
            print(subdf[['sample_id', 'visit', 'age_at_visit', 'ad', 'ftd', 'pd', 'als']])
    return subdfs_multi_visit


In [20]:
def multi_visit_by_disease(df_multi_visit, disease_col):
    disease_groups = []
    cohorts = []
    total_disease_count = len(disease_col)
    for df in df_multi_visit:
        count = 0
        for disease in disease_col:
            if (df[disease] == 1).any():
                count += 1
        if count == total_disease_count:
            disease_groups.append(df)
            cohorts.append(df["contributor_code"].iloc[0])
            if df["contributor_code"].nunique() > 1:
                raise ValueError("Cohort inconsistency within the same person_id.")
    print(f"Total persons with multiple visits and {disease_col}=1: {len(disease_groups)}")
    print("Different cohorts in this group:", set(cohorts))
    return disease_groups

In [21]:
def multi_visit_no_disease(df_multi_visit):
    disease_groups = []
    cohorts = []
    disease_col = ["ad", "ftd", "pd", "als", "mci_sci", "recruited_control"]
    total_disease_count = len(disease_col)
    for df in df_multi_visit:
        count = 0
        for disease in disease_col:
            if (df[disease] == 1).any():
                count += 1
        if count == 0:
            disease_groups.append(df)
            cohorts.append(df["contributor_code"].iloc[0])
            if df["contributor_code"].nunique() > 1:
                raise ValueError("Cohort inconsistency within the same person_id.")
    print(f"Total persons with multiple visits and no diagnoised disease: {len(disease_groups)}")
    print("Different cohorts in this group:", set(cohorts))
    return disease_groups

In [22]:
plasma_multi_visits = longitudinal_stats(plasma_with_clinical, mappingData)


Total unique persons: 18241
Total persons with multiple visits: 2552
Total number of multiple visits: 7154
Different cohorts in multi-visit persons: {'T', 'J', 'M', 'B', 'F', 'D', 'I', 'P', 'R'}


In [23]:
no_plasma_multi_visits = multi_visit_no_disease(plasma_multi_visits)

Total persons with multiple visits and no diagnoised disease: 1193
Different cohorts in this group: {'J', 'M', 'B', 'F', 'P', 'I', 'R'}


In [24]:
def sample_summary(df, cols = None):
    if cols is None:
        cols = ["person_id", "sample_id", "sex", "sample_type", "visit", "age_at_visit", "ad", "ftd", "pd", "als", "mci_sci", "recruited_control", "cdr", "cognitive_test_date", "cognitive_test_score", "computed_cognitive_test_score", "cognitive_test_battery", "computed_clinical_diagnosis", "computed_cognitive_impairment", "is_neuropath", "is_biomarker"]
    subdf = df[cols]
    return subdf


In [25]:
summary = sample_summary(no_plasma_multi_visits[0])
summary

,person_id,sample_id,sex,sample_type,visit,age_at_visit,ad,ftd,pd,als,...,recruited_control,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
2,000a58d9-feea-4ff8-95d7-fca3fa91e174,000a58d9-feea-4ff8-95d7-fca3fa91e174,1,Sample,1,57,0,0,0,0,...,0,0.0,11/4/2013,30,1,MMSE,0,0,1,0
3817,000a58d9-feea-4ff8-95d7-fca3fa91e174,2b0e352c-1d64-435e-894a-d15ce702889e,1,Sample,3,66,0,0,0,0,...,0,0.0,3/21/2022,30,1,MMSE,0,0,1,0
4240,000a58d9-feea-4ff8-95d7-fca3fa91e174,2fa1b0bf-ce6f-4afd-aa3b-ea27c1e891db,1,Sample,2,63,0,0,0,0,...,0,0.0,4/8/2019,30,1,MMSE,0,0,1,0


In [26]:
ad_plasma_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["ad"])

Total persons with multiple visits and ['ad']=1: 597
Different cohorts in this group: {'J', 'F', 'D', 'I', 'R'}


In [27]:
pd_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["pd"])

Total persons with multiple visits and ['pd']=1: 169
Different cohorts in this group: {'T', 'J', 'F', 'D', 'I', 'R'}


In [28]:
ftd_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["ftd"])

Total persons with multiple visits and ['ftd']=1: 19
Different cohorts in this group: {'T', 'M', 'I', 'F'}


In [29]:
als_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["als"])

Total persons with multiple visits and ['als']=1: 229
Different cohorts in this group: {'M'}


In [30]:
msi_sci_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["mci_sci"])

Total persons with multiple visits and ['mci_sci']=1: 589
Different cohorts in this group: {'J', 'B', 'D', 'I', 'P', 'R'}


In [31]:
control_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["recruited_control"])

Total persons with multiple visits and ['recruited_control']=1: 0
Different cohorts in this group: set()


In [32]:
ad_plasma_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["ad", "mci_sci"])

Total persons with multiple visits and ['ad', 'mci_sci']=1: 196
Different cohorts in this group: {'D', 'I', 'R', 'J'}


In [33]:
ad_plasma_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["pd", "mci_sci"])

Total persons with multiple visits and ['pd', 'mci_sci']=1: 20
Different cohorts in this group: {'D', 'I', 'R', 'J'}


In [34]:
ad_plasma_multi_visits = multi_visit_by_disease(plasma_multi_visits, ["ad", "pd"])

Total persons with multiple visits and ['ad', 'pd']=1: 27
Different cohorts in this group: {'J', 'F', 'D', 'I', 'R'}


In [35]:
csf_multi_visits = longitudinal_stats(csf_with_clinical, mappingData)


Total unique persons: 3187
Total persons with multiple visits: 41
Total number of multiple visits: 82
Different cohorts in multi-visit persons: {'T', 'J'}


In [36]:
serum_multi_visits = longitudinal_stats(serum_with_clinical, mappingData)


Total unique persons: 1676
Total persons with multiple visits: 1258
Total number of multiple visits: 3794
Different cohorts in multi-visit persons: {'U'}


In [37]:
pd_serum_multi_visits = multi_visit_by_disease(serum_multi_visits, ["pd"])

Total persons with multiple visits and ['pd']=1: 1258
Different cohorts in this group: {'U'}


In [38]:
citrate_multi_visits = longitudinal_stats(citrate_with_clinical, mappingData)


Total unique persons: 537
Total persons with multiple visits: 101
Total number of multiple visits: 202
Different cohorts in multi-visit persons: {'E'}


In [39]:
combined_df = pd.concat(plasma_multi_visits, ignore_index=True)

In [40]:
def remove_columns_with_missing_values(combined_df):
    """
    Remove seq_ columns that contain -1 values.
    Args:
        combined_df (pd.DataFrame): DataFrame containing the data.
    Returns:
        pd.DataFrame: DataFrame with specified columns removed.
    """
    # Assume combined_df is your DataFrame

    # 1️⃣ Get all seq_ columns
    seq_cols = [c for c in combined_df.columns if c.startswith("seq_")]

    # 2️⃣ Detect seq_ columns that contain -1
    seq_cols_with_minus1 = [c for c in seq_cols if (combined_df[c] == -1).any()]

    # 3️⃣ Count them
    num_dropped = len(seq_cols_with_minus1)
    print(f"Number of seq_ columns dropped because of missing values (-1): {num_dropped}")

    # 4️⃣ Keep the rest of the columns
    df_cleaned = combined_df.drop(columns=seq_cols_with_minus1)
    return df_cleaned

In [41]:
clean_df = remove_columns_with_missing_values(combined_df)

Number of seq_ columns dropped because of missing values (-1): 0


In [42]:
clean_df["is_somalogic"].value_counts()

is_somalogic
1    7154
Name: count, dtype: int64

In [43]:
clean_df["is_mass_spec"].value_counts()

is_mass_spec
0    7154
Name: count, dtype: int64

In [44]:
def filter_data_complete_entries(df):
    '''
    Keep only rows where all seq_ columns have valid data (not -1).
    Args:
        df (pd.DataFrame): DataFrame containing the data.
    Returns:
        pd.DataFrame: Filtered DataFrame with complete entries.
    '''
    # 1️⃣ Get all seq_ columns
    seq_cols = [c for c in df.columns if c.startswith("seq_")]

    # 2️⃣ Keep rows where none of the seq_ columns is -1
    df_filtered = df[(df[seq_cols] != -1).all(axis=1)]

    print(f"Number of entries after filtering for complete data: {len(df_filtered)}")
    return df_filtered

In [46]:
plasma_7k_with_clinical = filter_data_complete_entries(plasma_with_clinical)
print(plasma_7k_with_clinical.shape)
plasma_7k_with_clinical.head()

Number of entries after filtering for complete data: 19249
(19249, 7344)


,sample_id,contributor_code,visit,sample_type,seq_10000_28,seq_10001_7,seq_10003_15,seq_10006_25,seq_10008_43,seq_10010_10,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,0003610c-e435-44f3-83b4-d40dd750a13f,C,1,Sample,457.2,385.2,149.8,719.2,403.7,307.5,...,-1,-1.0,11/18/2019,28,1,MMSE,-1,0,0,1
1,00090ab1-6e72-4b1f-8a38-c99dda26d7dd,G,1,Sample,500.7,292.5,205.2,959.3,547.0,317.5,...,-1,-1.0,1/1/1900,29,1,MMSE,-1,0,-1,-1
2,000a58d9-feea-4ff8-95d7-fca3fa91e174,B,1,Sample,487.0,313.6,158.4,553.7,453.7,299.2,...,0,0.0,11/4/2013,30,1,MMSE,0,0,1,0
3,000b14dc-ba73-46ee-a69e-e681ca3ad857,F,1,Sample,611.6,250.4,170.3,653.3,479.1,353.7,...,0,1.0,1/1/1900,-1,-1,MMSE,1,1,0,-1
4,001016d4-a28d-4a66-9c88-92b8a7a04d2d,F,1,Sample,602.3,254.2,189.0,638.8,452.4,463.5,...,0,1.0,1/1/1900,16,1,MMSE,1,1,0,-1
